### Test API Connetion with Traffic NZ

In [1]:
import requests

URL = "https://trafficnz.info/service/traffic/rest/4/events/all/10"

resp = requests.get(URL, timeout=30)
print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Size (KB):", round(len(resp.content) / 1024, 1))

data = resp.json()
print("Top-level keys:", list(data.keys()))
print("Inner keys:", list(data["response"].keys()))
print("Number of events:", len(data["response"]["roadevent"]))

Status: 200
Content-Type: application/json
Size (KB): 350.9
Top-level keys: ['response']
Inner keys: ['roadevent']
Number of events: 245


API have 245 events across New Zealand roadworks 

### FIlter only WLG Region

In [2]:
import pandas as pd

events = data["response"]["roadevent"]
df = pd.json_normalize(events)

print("Total columns:", len(df.columns))
print("Regions in feed:")
print(df["region.name"].value_counts(dropna=False))

wgtn = df[df["region.name"] == "Wellington"].copy()
print("\nWellington events:", len(wgtn))
print("\nBy status:")
print(wgtn["status"].value_counts(dropna=False))
print("\nBy eventType:")
print(wgtn["eventType"].value_counts(dropna=False))
print("\nBy planned:")
print(wgtn["planned"].value_counts(dropna=False))

Total columns: 43
Regions in feed:
region.name
Waikato               65
Auckland              29
Bay Of Plenty         23
Nelson/Marlborough    22
Southland             16
West Coast            14
Wellington            13
Manawatu-Whanganui    12
Gisborne              12
Hawkes Bay            11
Canterbury             9
Taranaki               7
Otago                  6
Northland              6
Name: count, dtype: int64

Wellington events: 13

By status:
status
Active       6
Resolved     5
Scheduled    2
Name: count, dtype: int64

By eventType:
eventType
Road Work              6
Area Warning           4
Scheduled Road Work    2
Road Hazard            1
Name: count, dtype: int64

By planned:
planned
True     12
False     1
Name: count, dtype: int64


Wellington currently has 13 events (6 Active, 5 Resolved, 2 Scheduled), so the dataset is quite small. This is why continuous polling—to build up a history—is more important than the snapshot itself. The 5 "Resolved" events are still present in the feed, just as I suspected; they need to be filtered out or tracked based on their status.

**Explore eventComments description from API**

In [3]:
col_info = pd.DataFrame({
    "dtype": wgtn.dtypes.astype(str),
    "non_null": wgtn.notna().sum(),
})
print(col_info.to_string())


print("\n" + "=" * 80)
for _, r in wgtn.sort_values("startDate").iterrows():
    print(f"[{r['id']}] {r['status']} | {r['eventType']} | {r['eventDescription']} | impact={r['impact']} | planned={r['planned']}")
    print(f"  Where : {r['locationArea']}")
    print(f"  When  : {r['startDate']} -> {r.get('endDate')}")
    print(f"  Detour: {r.get('alternativeRoute')}")
    print(f"  Text  : {r['eventComments']}")
    print("-" * 80)

                                  dtype  non_null
alternativeRoute                 object        13
endDate                          object        12
eventComments                    object        13
eventCreated                     object        13
eventDescription                 object        13
eventIsland                      object        13
eventModified                    object        13
eventType                        object        13
expectedResolution               object        13
geometry                         object        13
id                                int64        13
impact                           object        13
informationSource                object        13
locationArea                     object        13
locations                        object        13
planned                            bool        13
startDate                        object        13
status                           object        13
supplier                         object        13


### classify closure types

In [4]:
text = wgtn["eventComments"].fillna("").str.lower()

wgtn["is_cleared"] = text.str.contains(r"now clear", regex=True)
wgtn["is_road_closed"] = (
    (wgtn["impact"] == "Road Closed")
    | text.str.contains(
        r"\b(?:road|ramp|highway|bridge|tunnel)\b[^.]{0,40}\b(?:closed|close|closure)\b",
        regex=True,
    )
)
wgtn["is_lane_closure"] = text.str.contains(r"\blanes?\b[^.]{0,20}\bclos", regex=True)
wgtn["is_stop_go"] = text.str.contains(r"stop\s*/\s*go", regex=True)
wgtn["has_speed_restriction"] = text.str.contains(r"speed restriction|\d+\s*km/h", regex=True)

def primary_type(r):
    if r["is_cleared"]:
        return "cleared_no_detail"
    if r["is_road_closed"]:
        return "road_closure"
    if r["is_lane_closure"]:
        return "lane_closure"
    if r["is_stop_go"]:
        return "stop_go"
    if r["has_speed_restriction"]:
        return "speed_restriction"
    return "other"

wgtn["closure_type"] = wgtn.apply(primary_type, axis=1)

print(wgtn["closure_type"].value_counts())
print("\nimpact vs closure_type:")
print(pd.crosstab(wgtn["impact"], wgtn["closure_type"]))

wgtn[["id", "status", "impact", "closure_type", "is_stop_go",
      "has_speed_restriction", "locationArea"]].sort_values("closure_type")

closure_type
stop_go              8
road_closure         3
cleared_no_detail    2
Name: count, dtype: int64

impact vs closure_type:
closure_type  cleared_no_detail  road_closure  stop_go
impact                                                
Caution                       2             2        7
Delays                        0             0        1
Road Closed                   0             1        0


,id,status,impact,closure_type,is_stop_go,has_speed_restriction,locationArea
69,565023,Resolved,Caution,cleared_no_detail,False,False,"SH 2 Petone, at Petone offramp"
99,564954,Resolved,Caution,cleared_no_detail,False,False,SH 2 Clareville between Hughes Line and Cheste...
65,564875,Scheduled,Caution,road_closure,False,False,SH 1 Newlands Northbound On-Ramp
66,564876,Scheduled,Caution,road_closure,False,False,SH 1 Newlands Southbound On-Ramp
76,564146,Active,Road Closed,road_closure,False,False,SH 2 Kaitoke to Featherston (Remutaka Hill)
70,565053,Active,Caution,stop_go,True,True,"SH 53 Featherston, between Lyon Street and Dan..."
71,564971,Resolved,Caution,stop_go,True,True,"SH 53 Featherston, between Lyon Street and Wak..."
72,563855,Active,Caution,stop_go,True,True,"SH 58 Judgeford, between Flightys Road and Moo..."
73,565061,Active,Caution,stop_go,True,True,SH 2 Totara Park Intersection
74,564965,Resolved,Caution,stop_go,True,True,"SH 2 Totara Park, at Totara Park Road intersec..."


### parse the geometry into lat/lon

In [5]:
import re

def parse_wkt(wkt):
    """Return (geom_type, parts) where each part is a list of (lon, lat) tuples."""
    geom_type = wkt.split("(")[0].strip().upper()
    parts = []
    for chunk in re.findall(r"\(([^()]+)\)", wkt):
        pts = []
        for pair in chunk.split(","):
            lon, lat = pair.strip().split()[:2]
            pts.append((float(lon), float(lat)))
        parts.append(pts)
    return geom_type, parts

def mean_point(parsed_geom):
    pts = [pt for part in parsed_geom[1] for pt in part]
    return (sum(p[0] for p in pts) / len(pts), sum(p[1] for p in pts) / len(pts))

parsed = wgtn["geometry"].apply(parse_wkt)
wgtn["geom_type"] = parsed.apply(lambda g: g[0])
wgtn["n_vertices"] = parsed.apply(lambda g: sum(len(p) for p in g[1]))
wgtn["lon"] = parsed.apply(lambda g: mean_point(g)[0])
wgtn["lat"] = parsed.apply(lambda g: mean_point(g)[1])

print(wgtn["geom_type"].value_counts())

out_of_range = wgtn[~wgtn["lat"].between(-42, -40) | ~wgtn["lon"].between(174, 176.5)]
print("\nRows outside the expected Wellington lat/lon range:", len(out_of_range))

wgtn[["id", "geom_type", "n_vertices", "lat", "lon", "locationArea"]]

geom_type
POINT              9
MULTILINESTRING    4
Name: count, dtype: int64

Rows outside the expected Wellington lat/lon range: 0


,id,geom_type,n_vertices,lat,lon,locationArea
65,564875,POINT,1,-41.232674,174.807725,SH 1 Newlands Northbound On-Ramp
66,564876,POINT,1,-41.232260,174.809654,SH 1 Newlands Southbound On-Ramp
69,565023,POINT,1,-41.224258,174.857310,"SH 2 Petone, at Petone offramp"
70,565053,POINT,1,-41.118267,175.324859,"SH 53 Featherston, between Lyon Street and Dan..."
71,564971,POINT,1,-41.118030,175.324535,"SH 53 Featherston, between Lyon Street and Wak..."
72,563855,MULTILINESTRING,3,-41.117390,174.944003,"SH 58 Judgeford, between Flightys Road and Moo..."
73,565061,POINT,1,-41.115279,175.081348,SH 2 Totara Park Intersection
74,564965,POINT,1,-41.115246,175.081310,"SH 2 Totara Park, at Totara Park Road intersec..."
75,564136,MULTILINESTRING,3,-41.105773,174.901950,"SH 58 Whitby, between the intersections of Pos..."
76,564146,MULTILINESTRING,16,-41.105925,175.245597,SH 2 Kaitoke to Featherston (Remutaka Hill)


In [8]:
import folium

COLOURS = {
    "road_closure": "red",
    "lane_closure": "purple",
    "stop_go": "orange",
    "speed_restriction": "blue",
    "cleared_no_detail": "gray",
    "other": "black",
}

m = folium.Map(location=[-41.15, 175.0], zoom_start=9, tiles="OpenStreetMap")

for (_, r), (geom_type, parts) in zip(wgtn.iterrows(), parsed):
    colour = COLOURS[r["closure_type"]]
    end = r["endDate"] if pd.notna(r["endDate"]) else "no end date"
    popup = folium.Popup(
        f"<b>{r['locationArea']}</b><br>"
        f"{r['closure_type']} | {r['status']}<br>"
        f"{r['eventDescription']} | impact: {r['impact']}<br>"
        f"{r['startDate'][:16]} → {str(end)[:16]}",
        max_width=300,
    )
    if geom_type == "POINT":
        lon, lat = parts[0][0]
        folium.CircleMarker(
            [lat, lon], radius=8, color=colour,
            fill=True, fill_opacity=0.8, popup=popup,
        ).add_to(m)
    else:
        for part in parts:
            folium.PolyLine(
                [(lat, lon) for lon, lat in part],
                color=colour, weight=5, popup=popup,
            ).add_to(m)

m.save("wlg-events-map.html")
m

The map confirms all three checks.

Remutaka: the red line follows SH2 over the hill between Upper Hutt and Featherston.
Newlands: the red dot sits north of Wellington city near Johnsonville. The two ramps are about 150 m apart, so they draw as one dot.
Cleared events: the grey dots are at Petone and near Carterton, which is where Clareville is.

The map also shows two things to keep in mind. Resolved events like the SH53 and Totara Park ones are still drawn in orange, so the dashboard needs a status filter or dimmed styling. And some events share a location (Featherston, Totara Park, Masterton), so their markers overlap and we’ll want clustering.


<table>
  <thead>
    <tr>
      <th>Table</th>
      <th>Purpose</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>raw_road_events</code></td>
      <td>
        Append-only history. One row per version of an event,
        with the full JSON stored in a jsonb column.
        Primary key: (event_id, payload_md5).
      </td>
    </tr>
    <tr>
      <td><code>ingest_runs</code></td>
      <td>
        One row per poll, tracking poll time, feed events,
        Wellington events, new versions inserted, and
        success/error status. Powers /healthz and
        freshness alerts.
      </td>
    </tr>
    <tr>
      <td><code>dbt models (later)</code></td>
      <td>
        stg_road_events with parsed fields, closure flags,
        latitude/longitude, and WKT. Downstream marts
        include current active events and event duration.
      </td>
    </tr>
  </tbody>
</table>


Two decisions are built into it:

Filter to Wellington at ingest. Storing all 245 NZ events every poll would add roughly 16 MB a day. Wellington is about 13 events, and we insert only when an event’s content has changed, so it stays tiny on Neon’s free tier.
Deduplicate on a hash of the payload. I haven’t checked that eventModified changes on every edit, so the hash doesn’t depend on that assumption. It’s the same idea as the em6 lesson: don’t trust the source to signal change.

In [10]:
import re

def highway(r):
    m = re.search(r"\bSH\s?(\d+)", str(r["locationArea"]))
    return f"SH{m.group(1)}" if m else r["journey.name"]

wgtn["highway"] = wgtn.apply(highway, axis=1)
wgtn["is_roadwork"] = wgtn["eventType"] != "Road Hazard"

print("Wellington region events:", len(wgtn))
print("Roadworks (excl. Road Hazard):", wgtn["is_roadwork"].sum())

print("\nStatus x roadwork:")
print(pd.crosstab(wgtn["status"], wgtn["is_roadwork"]))

print("\nBy highway:")
print(wgtn.groupby("highway").size())

print("\nRoadworks by highway and location:")
rw = wgtn[wgtn["is_roadwork"]].sort_values(["highway", "status", "locationArea"])
print(rw[["highway", "locationArea", "status", "eventType", "eventDescription"]].to_string(index=False))

# Does anything in the whole NZ feed lack an "SH <number>" in its location?
non_sh = df[~df["locationArea"].str.contains(r"\bSH\s?\d+", regex=True, na=False)]
print("\nNZ-wide events without 'SH <number>' in locationArea:", len(non_sh))
print(non_sh[["region.name", "locationArea", "way.name"]].to_string(index=False))

Wellington region events: 13
Roadworks (excl. Road Hazard): 12

Status x roadwork:
is_roadwork  False  True 
status                   
Active           0      6
Resolved         1      4
Scheduled        0      2

By highway:
highway
SH1     2
SH2     7
SH53    2
SH58    2
dtype: int64

Roadworks by highway and location:
highway                                                                     locationArea    status           eventType eventDescription
    SH1                                                 SH 1 Newlands Northbound On-Ramp Scheduled Scheduled Road Work      Resurfacing
    SH1                                                 SH 1 Newlands Southbound On-Ramp Scheduled Scheduled Road Work      Resurfacing
    SH2                                      SH 2 Kaitoke to Featherston (Remutaka Hill)    Active        Area Warning      Maintenance
    SH2                   SH 2 Masterton, between Fleet Street and Intermediate  Streets    Active        Area Warning       Road Wor

### inspect the WCC layer

In [11]:
import requests

WCC_URL = ("https://gis.wcc.govt.nz/arcgis/rest/services/Transportation/"
           "StreetEventsAndRoadClosures/MapServer/1/query")

params = {
    "where": "1=1",
    "outFields": "*",
    "outSR": 4326,          # ask the server to reproject to lat/lon
    "f": "json",
    "resultRecordCount": 5,
}

r = requests.get(WCC_URL, params=params, timeout=30)
print("Status:", r.status_code)
j = r.json()

if "error" in j:
    print("API error:", j["error"])
else:
    print("Geometry type:", j.get("geometryType"))
    print("Features returned:", len(j.get("features", [])))
    print("\nFields:")
    for f in j.get("fields", []):
        print(" ", f["name"], "-", f["type"])
    print("\nSample records:")
    for feat in j.get("features", []):
        print(feat["attributes"])
        print("-" * 60)

Status: 200
Geometry type: esriGeometryPolyline
Features returned: 5

Fields:
  OBJECTID - esriFieldTypeOID
  Event_Name - esriFieldTypeString
  Start_Date - esriFieldTypeDate
  End_Date - esriFieldTypeDate
  EventType - esriFieldTypeInteger
  Approved - esriFieldTypeSmallInteger
  EventDetails - esriFieldTypeString
  Shape.STLength() - esriFieldTypeDouble
  Symbology - esriFieldTypeString

Sample records:
{'OBJECTID': 196, 'Event_Name': 'Thorndon Fair', 'Start_Date': 1796536800000, 'End_Date': 1796580000000, 'EventType': 1, 'Approved': 1, 'EventDetails': 'Pedestrian access will not be restricted, and emergency services will have immediate access to the area if required. Public transport operators will been notified of the proposed closure ', 'Shape.STLength()': 636.1560190640582, 'Symbology': None}
------------------------------------------------------------
{'OBJECTID': 197, 'Event_Name': 'Newtown Festival', 'Start_Date': 1804395600000, 'End_Date': 1804453200000, 'EventType': 1, 'App

In [12]:
import pandas as pd
import requests

LAYER_URL = ("https://gis.wcc.govt.nz/arcgis/rest/services/Transportation/"
             "StreetEventsAndRoadClosures/MapServer/1")

# 1) Layer metadata: do the fields define coded-value domains (e.g. EventType)?
meta = requests.get(LAYER_URL, params={"f": "json"}, timeout=30).json()
for f in meta.get("fields", []):
    if f.get("domain"):
        print(f["name"], "domain:", f["domain"])

# 2) All records, attributes only (no geometry)
r = requests.get(
    f"{LAYER_URL}/query",
    params={"where": "1=1", "outFields": "*", "returnGeometry": "false", "f": "json"},
    timeout=60,
)
r.raise_for_status()
wcc = pd.DataFrame([f["attributes"] for f in r.json()["features"]])

for col in ["Start_Date", "End_Date"]:
    wcc[col] = (pd.to_datetime(wcc[col], unit="ms", utc=True)
                  .dt.tz_convert("Pacific/Auckland"))

now = pd.Timestamp.now(tz="Pacific/Auckland")

print("\nTotal records:", len(wcc))
print("EventType values:\n", wcc["EventType"].value_counts(dropna=False))
print("Approved values:\n", wcc["Approved"].value_counts(dropna=False))
print("Date range:", wcc["Start_Date"].min(), "->", wcc["End_Date"].max())
print("Past:", (wcc["End_Date"] < now).sum(),
      "| Current:", ((wcc["Start_Date"] <= now) & (wcc["End_Date"] >= now)).sum(),
      "| Upcoming:", (wcc["Start_Date"] > now).sum())

wcc[wcc["End_Date"] >= now].sort_values("Start_Date")[
    ["OBJECTID", "Event_Name", "Start_Date", "End_Date", "EventType", "Approved"]
]

EventType domain: {'type': 'codedValue', 'name': 'Event_Type', 'description': 'Event Type', 'codedValues': [{'name': 'Road Closure', 'code': 1}, {'name': 'Parade', 'code': 2}, {'name': 'Parade and Road Closure', 'code': 3}, {'name': 'Diversion', 'code': 4}], 'mergePolicy': 'esriMPTDefaultValue', 'splitPolicy': 'esriSPTDefaultValue'}
Approved domain: {'type': 'codedValue', 'name': 'Approved', 'description': 'Is this event Approved or Not?', 'codedValues': [{'name': 'Yes', 'code': 1}, {'name': 'No', 'code': 2}], 'mergePolicy': 'esriMPTDefaultValue', 'splitPolicy': 'esriSPTDefaultValue'}

Total records: 60
EventType values:
 EventType
1.0    51
NaN     7
2.0     1
3.0     1
Name: count, dtype: int64
Approved values:
 Approved
1.0    35
NaN    25
Name: count, dtype: int64
Date range: 2021-02-20 23:00:00+13:00 -> 2027-07-05 01:00:00+12:00
Past: 41 | Current: 1 | Upcoming: 16


,OBJECTID,Event_Name,Start_Date,End_Date,EventType,Approved
33,5423,Wilson Street Pop-Up,2026-07-01 18:00:00+12:00,2027-07-01 08:00:00+12:00,1.0,NaN
14,210,Polish Christmas Market,2026-11-14 19:00:00+13:00,2026-11-15 12:30:00+13:00,1.0,1.0
5,201,Blackcaps vs India 1st Test Match,2026-11-16 19:00:00+13:00,2026-11-25 10:00:00+13:00,1.0,NaN
0,196,Thorndon Fair,2026-12-06 19:00:00+13:00,2026-12-07 07:00:00+13:00,1.0,1.0
12,208,Rotary Khandallah Fair,2026-12-13 19:00:00+13:00,2026-12-14 05:00:00+13:00,1.0,1.0
50,14235,Courtenay Carnival,2026-12-31 19:00:00+13:00,2027-01-01 19:00:00+13:00,1.0,1.0
23,1019,Domestic T20 Cricket Series,2027-01-02 19:00:00+13:00,2027-01-26 10:00:00+13:00,1.0,1.0
10,206,Day in the Bay,2027-02-07 19:00:00+13:00,2027-02-08 08:00:00+13:00,1.0,1.0
55,15424,Paniyiri (Food Festival),2027-02-27 22:00:00+13:00,2027-02-28 04:00:00+13:00,1.0,1.0
1,197,Newtown Festival,2027-03-07 18:00:00+13:00,2027-03-08 10:00:00+13:00,1.0,1.0


### Test Connections

In [1]:
import os
import psycopg2
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

conn = psycopg2.connect(os.environ["NEON_DATABASE_URL"])
with conn.cursor() as cur:
    cur.execute("select current_database(), version(), now()")
    print(cur.fetchone())
conn.close()

('neondb', 'PostgreSQL 18.6 (fd34222) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit', datetime.datetime(2026, 10, 25, 21, 15, 46, 827571, tzinfo=datetime.timezone.utc))


### which fields change between polls

In [2]:
import time
import json
import requests

URL = "https://trafficnz.info/service/traffic/rest/4/events/all/10"

def fetch_wgtn():
    r = requests.get(URL, timeout=30)
    r.raise_for_status()
    evs = r.json()["response"]["roadevent"]
    return {e["id"]: e for e in evs if (e.get("region") or {}).get("name") == "Wellington"}

def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, f"{prefix}{k}."))
        else:
            out[f"{prefix}{k}"] = json.dumps(v, sort_keys=True)
    return out

snap_a = fetch_wgtn()
time.sleep(120)
snap_b = fetch_wgtn()

changed = {}
for eid in snap_a.keys() & snap_b.keys():
    fa, fb = flatten(snap_a[eid]), flatten(snap_b[eid])
    for k in fa.keys() | fb.keys():
        if fa.get(k) != fb.get(k):
            changed.setdefault(k, []).append(eid)

print("Only in first call :", snap_a.keys() - snap_b.keys())
print("Only in second call:", snap_b.keys() - snap_a.keys())
print("Fields that changed:", changed or "none")

Only in first call : set()
Only in second call: set()
Fields that changed: none


### Create 3 tables

In [3]:
import os
import psycopg2
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

def get_conn():
    return psycopg2.connect(os.environ["NEON_DATABASE_URL"])

DDL = """
create table if not exists ingest_runs (
    run_id            bigserial primary key,
    started_at        timestamptz not null default now(),
    finished_at       timestamptz,
    success           boolean,
    events_in_feed    int,
    wellington_events int,
    new_events        int,
    changed_events    int,
    unchanged_events  int,
    gone_events       int,
    error             text
);

create table if not exists road_events_current (
    event_id         bigint primary key,
    region           text,
    status           text,
    content_hash     text not null,
    payload          jsonb not null,
    first_seen_at    timestamptz not null,
    last_seen_at     timestamptz not null,
    last_changed_at  timestamptz not null,
    in_feed          boolean not null default true
);

create table if not exists road_event_versions (
    version_id     bigserial primary key,
    event_id       bigint not null,
    content_hash   text not null,
    status         text,
    payload        jsonb not null,
    observed_at    timestamptz not null default now(),
    ingest_run_id  bigint references ingest_runs(run_id)
);

create index if not exists idx_versions_event
    on road_event_versions (event_id, observed_at);
"""

conn = get_conn()
with conn.cursor() as cur:
    cur.execute(DDL)
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        select table_name from information_schema.tables
        where table_schema = 'public' order by table_name
    """)
    print([r[0] for r in cur.fetchall()])
conn.close()

['ingest_runs', 'road_event_versions', 'road_events_current']


### The ingest function (hash, compare, upsert)

In [4]:
import hashlib
import json
import requests
from psycopg2.extras import Json

URL = "https://trafficnz.info/service/traffic/rest/4/events/all/10"
REGION = "Wellington"

# Only fields that describe the event itself. Live traffic metrics under
# "journeyLeg" and the bookkeeping field "eventModified" are deliberately left out.
HASH_FIELDS = [
    "id", "status", "eventType", "eventDescription", "eventComments",
    "impact", "startDate", "endDate", "expectedResolution", "geometry",
    "locationArea", "locations", "alternativeRoute", "planned",
    "restrictions", "informationSource", "supplier",
]

def content_hash(e):
    subset = {k: e.get(k) for k in HASH_FIELDS}
    subset["region"] = (e.get("region") or {}).get("name")
    raw = json.dumps(subset, sort_keys=True, default=str)
    return hashlib.sha256(raw.encode()).hexdigest()

def run_ingest(conn):
    # 1) Open a run record and commit it, so a failure is still logged
    with conn.cursor() as cur:
        cur.execute("insert into ingest_runs default values returning run_id")
        run_id = cur.fetchone()[0]
    conn.commit()

    stats = dict(events_in_feed=0, wellington_events=0, new_events=0,
                 changed_events=0, unchanged_events=0, gone_events=0)
    try:
        # 2) Fetch the whole NZ feed, keep only Wellington
        resp = requests.get(URL, timeout=30)
        resp.raise_for_status()
        all_events = resp.json()["response"]["roadevent"]
        stats["events_in_feed"] = len(all_events)
        if not all_events:
            # Guard: an empty feed must not mark every event as gone
            raise RuntimeError("Feed returned 0 events; refusing to continue")

        wgtn = {e["id"]: e for e in all_events
                if (e.get("region") or {}).get("name") == REGION}
        stats["wellington_events"] = len(wgtn)

        with conn.cursor() as cur:
            cur.execute("select event_id, content_hash from road_events_current")
            existing = dict(cur.fetchall())

            for eid, e in wgtn.items():
                h = content_hash(e)
                status = e.get("status")

                if eid not in existing:
                    # NEW event: first version + current row
                    cur.execute("""
                        insert into road_event_versions
                            (event_id, content_hash, status, payload, ingest_run_id)
                        values (%s, %s, %s, %s, %s)""",
                        (eid, h, status, Json(e), run_id))
                    cur.execute("""
                        insert into road_events_current
                            (event_id, region, status, content_hash, payload,
                             first_seen_at, last_seen_at, last_changed_at, in_feed)
                        values (%s, %s, %s, %s, %s, now(), now(), now(), true)""",
                        (eid, REGION, status, h, Json(e)))
                    stats["new_events"] += 1

                elif existing[eid] != h:
                    # CHANGED event: append a version, refresh the current row
                    cur.execute("""
                        insert into road_event_versions
                            (event_id, content_hash, status, payload, ingest_run_id)
                        values (%s, %s, %s, %s, %s)""",
                        (eid, h, status, Json(e), run_id))
                    cur.execute("""
                        update road_events_current
                        set status = %s, content_hash = %s, payload = %s,
                            last_seen_at = now(), last_changed_at = now(),
                            in_feed = true
                        where event_id = %s""",
                        (status, h, Json(e), eid))
                    stats["changed_events"] += 1

                else:
                    # UNCHANGED event: just record that we saw it again
                    cur.execute("""
                        update road_events_current
                        set last_seen_at = now(), in_feed = true
                        where event_id = %s""", (eid,))
                    stats["unchanged_events"] += 1

            # GONE: seen before, still marked in_feed, but absent from this snapshot
            gone_ids = [eid for eid in existing if eid not in wgtn]
            if gone_ids:
                cur.execute("""
                    update road_events_current set in_feed = false
                    where event_id = any(%s) and in_feed""", (gone_ids,))
                stats["gone_events"] = cur.rowcount

            cur.execute("""
                update ingest_runs
                set finished_at = now(), success = true,
                    events_in_feed = %s, wellington_events = %s,
                    new_events = %s, changed_events = %s,
                    unchanged_events = %s, gone_events = %s
                where run_id = %s""",
                (stats["events_in_feed"], stats["wellington_events"],
                 stats["new_events"], stats["changed_events"],
                 stats["unchanged_events"], stats["gone_events"], run_id))
        conn.commit()

    except Exception as ex:
        conn.rollback()
        with conn.cursor() as cur:
            cur.execute("""
                update ingest_runs
                set finished_at = now(), success = false, error = %s,
                    events_in_feed = %s
                where run_id = %s""",
                (str(ex)[:1000], stats["events_in_feed"], run_id))
        conn.commit()
        raise

    stats["run_id"] = run_id
    return stats

print("run_ingest defined")

run_ingest defined


In [7]:
from IPython.display import HTML, display

html = """
<style>
.traffic-flow {
    max-width: 1050px;
    margin: 20px auto;
    padding: 30px;
    background: #ffffff;
    border: 1px solid #dddddd;
    border-radius: 22px;
    font-family: Arial, sans-serif;
    color: #222222;
    box-sizing: border-box;
}
.traffic-step {
    text-align: center;
    padding: 12px 10px;
}
.traffic-step h3 {
    font-size: 21px;
    font-weight: 700;
    margin: 8px 0;
}
.traffic-step p {
    color: #666666;
    font-size: 15px;
    margin: 8px 0;
}
.traffic-icon {
    font-size: 28px;
    color: #2563eb;
}
.traffic-arrow {
    text-align: center;
    font-size: 25px;
    color: #999999;
    margin: 12px 0;
}
.traffic-status-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 12px;
    margin-top: 15px;
}
.traffic-status {
    border: 1px solid #dddddd;
    border-radius: 12px;
    padding: 15px;
    background: #fafafa;
}
.traffic-status h4 {
    margin: 0 0 8px 0;
    font-size: 19px;
    font-weight: 700;
}
.traffic-status p {
    margin: 0;
    font-size: 14px;
    color: #666666;
}
.traffic-new { color: #16a34a; }
.traffic-changed { color: #ea580c; }
.traffic-unchanged { color: #666666; }
.traffic-gone { color: #dc2626; }

.traffic-db-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 12px;
    margin-top: 20px;
}
.traffic-db {
    padding: 14px;
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    min-width: 0;
}
.traffic-db h4 {
    font-size: 14px;
    margin: 0 0 8px 0;
    overflow-wrap: anywhere;
}
.traffic-db p {
    font-size: 13px;
    color: #64748b;
    margin: 0;
}
@media (max-width: 600px) {
    .traffic-status-grid,
    .traffic-db-grid {
        grid-template-columns: 1fr;
    }
}
</style>

<div class="traffic-flow">

    <!-- STEP 1: FETCH API DATA -->
    <div class="traffic-step">
        <div class="traffic-icon">&#9729;</div>
        <h3>Traffic NZ API</h3>
        <p>Fetch all road traffic events across New Zealand</p>
    </div>

    <div class="traffic-arrow">&darr;</div>

    <!-- STEP 2: FILTER WELLINGTON EVENTS -->
    <div class="traffic-step">
        <h3>Filter: Wellington</h3>
        <p>Keep only events where region.name = Wellington</p>
    </div>

    <div class="traffic-arrow">&darr;</div>

    <!-- STEP 3: GENERATE CONTENT HASH -->
    <div class="traffic-step">
        <h3>SHA-256 Content Hash</h3>
        <p>Generate a unique content fingerprint for each event</p>
    </div>

    <div class="traffic-arrow">&darr;</div>

    <!-- STEP 4: CLASSIFY EVENTS -->
    <div class="traffic-status-grid">

        <div class="traffic-status">
            <h4 class="traffic-new">NEW</h4>
            <p>Event does not exist in the database</p>
        </div>

        <div class="traffic-status">
            <h4 class="traffic-changed">CHANGED</h4>
            <p>Event content differs from the stored version</p>
        </div>

        <div class="traffic-status">
            <h4 class="traffic-unchanged">UNCHANGED</h4>
            <p>Event content remains the same</p>
        </div>

        <div class="traffic-status">
            <h4 class="traffic-gone">GONE</h4>
            <p>Event is no longer present in the API feed</p>
        </div>

    </div>

    <div class="traffic-arrow">&darr;</div>

    <!-- STEP 5: STORE DATA IN POSTGRESQL -->
    <div class="traffic-db-grid">

        <div class="traffic-db">
            <h4>road_events_current</h4>
            <p>Stores the latest state of each event</p>
        </div>

        <div class="traffic-db">
            <h4>road_event_versions</h4>
            <p>Maintains historical event versions</p>
        </div>

        <div class="traffic-db">
            <h4>ingest_runs</h4>
            <p>Tracks ingestion runs and pipeline status</p>
        </div>

    </div>

</div>
"""

display(HTML(html))

### First Run (Test)

In [8]:
conn = get_conn()
stats = run_ingest(conn)
print(stats)

with conn.cursor() as cur:
    cur.execute("select count(*), count(*) filter (where in_feed) from road_events_current")
    print("current rows / in_feed:", cur.fetchone())
    cur.execute("select count(*) from road_event_versions")
    print("version rows:", cur.fetchone()[0])
conn.close()

{'events_in_feed': 222, 'wellington_events': 7, 'new_events': 7, 'changed_events': 0, 'unchanged_events': 0, 'gone_events': 0, 'run_id': 1}
current rows / in_feed: (7, 7)
version rows: 7


### Second Run (After) - Test

In [9]:
conn = get_conn()
stats = run_ingest(conn)
print(stats)

with conn.cursor() as cur:
    cur.execute("select count(*) from road_event_versions")
    print("version rows:", cur.fetchone()[0])
conn.close()

{'events_in_feed': 222, 'wellington_events': 7, 'new_events': 0, 'changed_events': 0, 'unchanged_events': 7, 'gone_events': 0, 'run_id': 2}
version rows: 7


### tamper test (changed and gone paths)

This simulates two things: a real event whose content changed, and an event that disappeared. The cleanup at the end removes the test rows.

In [11]:
conn = get_conn()
with conn.cursor() as cur:
    # Corrupt the stored hash of one real event -> next run must see "changed"
    cur.execute("select event_id from road_events_current where in_feed limit 1")
    real_id = cur.fetchone()[0]
    cur.execute("update road_events_current set content_hash = 'tampered' where event_id = %s", (real_id,))

    # Insert a fake event that is not in the feed -> next run must see "gone"
    cur.execute("""
        insert into road_events_current
            (event_id, region, status, content_hash, payload,
             first_seen_at, last_seen_at, last_changed_at, in_feed)
        values (999999999, 'Wellington', 'Active', 'fake', '{}'::jsonb,
                now(), now(), now(), true)""")
conn.commit()

stats = run_ingest(conn)
print(stats)   # expect changed_events = 1 and gone_events = 1

{'events_in_feed': 222, 'wellington_events': 7, 'new_events': 0, 'changed_events': 1, 'unchanged_events': 6, 'gone_events': 1, 'run_id': 4}


In [12]:
with conn.cursor() as cur:
    cur.execute("select in_feed from road_events_current where event_id = 999999999")
    print("fake event in_feed:", cur.fetchone()[0])   # expect False

    # Cleanup: remove the fake event and the test version this run created
    cur.execute("delete from road_event_versions where ingest_run_id = %s", (stats["run_id"],))
    cur.execute("delete from road_events_current where event_id = 999999999")
conn.commit()
conn.close()

fake event in_feed: False


### look at what’s in the tables

In [13]:
import pandas as pd

conn = get_conn()
runs = pd.read_sql("select * from ingest_runs order by run_id", conn)
cur_rows = pd.read_sql("""
    select event_id, status, in_feed, first_seen_at, last_seen_at, last_changed_at,
           payload->>'locationArea' as location
    from road_events_current order by event_id""", conn)
conn.close()

print(runs.to_string(index=False))
print()
print(cur_rows.to_string(index=False))

 run_id                       started_at                      finished_at  success  events_in_feed  wellington_events  new_events  changed_events  unchanged_events  gone_events error
      1 2026-10-25 21:40:25.250843+00:00 2026-10-25 21:40:27.971777+00:00     True             222                  7           7               0                 0            0  None
      2 2026-10-25 21:40:34.234967+00:00 2026-10-25 21:40:37.040720+00:00     True             222                  7           0               0                 7            0  None
      3 2026-10-25 21:43:24.897027+00:00 2026-10-25 21:43:27.778456+00:00     True             222                  7           0               1                 6            1  None
      4 2026-10-25 21:43:49.551228+00:00 2026-10-25 21:43:52.064194+00:00     True             222                  7           0               1                 6            1  None

 event_id    status  in_feed                    first_seen_at                     la

/var/folders/qw/mns0z74d3t1_nqjy7zgm7mkw0000gn/T/ipykernel_40192/529992170.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs = pd.read_sql("select * from ingest_runs order by run_id", conn)
/var/folders/qw/mns0z74d3t1_nqjy7zgm7mkw0000gn/T/ipykernel_40192/529992170.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cur_rows = pd.read_sql("""


In [14]:
### measure the size of database
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("select pg_size_pretty(pg_database_size(current_database()))")
    print("Database size:", cur.fetchone()[0])

    cur.execute("""
        select relname,
               pg_size_pretty(pg_total_relation_size(relid)) as total_size,
               n_live_tup as approx_rows
        from pg_stat_user_tables
        order by pg_total_relation_size(relid) desc""")
    for row in cur.fetchall():
        print(row)

    cur.execute("select count(*) from road_event_versions")
    print("version rows:", cur.fetchone()[0])          # expect 7

    cur.execute("""select event_id, count(*) from road_event_versions
                   group by event_id having count(*) > 1""")
    print("events with >1 version:", cur.fetchall())   # expect []
conn.close()

Database size: 7888 kB
('road_events_current', '112 kB', 7)
('road_event_versions', '96 kB', 7)
('ingest_runs', '32 kB', 4)
version rows: 7
events with >1 version: []


In [15]:
# optional cleanup

conn = get_conn()
with conn.cursor() as cur:
    cur.execute("delete from road_event_versions where ingest_run_id in (3, 4)")
    cur.execute("delete from ingest_runs where run_id in (3, 4)")
conn.commit()
conn.close()

In [16]:
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("select count(*) from road_event_versions where ingest_run_id in (3, 4)")
    n = cur.fetchone()[0]
    print("versions tied to runs 3/4:", n)          # expect 0
    if n == 0:
        cur.execute("delete from ingest_runs where run_id in (3, 4)")
        conn.commit()
        print("deleted test runs 3 and 4")
    else:
        print("NOT deleting: those versions would be lost. Send me this output.")
conn.close()

versions tied to runs 3/4: 0
deleted test runs 3 and 4
